For this exercise we are considering the following model:

$$
\begin{align*}
\text{GAP} : \quad \min_{x} \quad & \sum_{i=1}^{n} \sum_{j=1}^{m} c_{ij}x_{ij} \\
\text{s.t.} \quad & \sum_{i=1}^{n} x_{ij} = 1 && j = 1, \dots, m \\
& \sum_{j=1}^{m} w_{ij}x_{ij} \le C_{i} && i = 1, \dots, n \\
& x_{ij} \in \{0, 1\} && i = 1, \dots, n , \ j = 1, \dots, m
\end{align*}
$$

### Lagrangian Relaxation 1: Dualizing the Assignment Constraints

In this approach, the constraints that each job must be assigned exactly once are relaxed and incorporated into the objective function with a vector of Lagrange multipliers $\lambda \in \mathbb{R}^m$.

The Lagrangian dual problem $L_1(\lambda)$ is:

$$
\begin{align*}
L_1(\lambda) : \quad \min_{x} \quad & \sum_{i=1}^{n} \sum_{j=1}^{m} c_{ij}x_{ij} + \sum_{j=1}^{m} \lambda_j \left(1 - \sum_{i=1}^{n} x_{ij}\right) \\
\text{s.t.} \quad & \sum_{j=1}^{m} w_{ij}x_{ij} \le C_{i} && i = 1, \dots, n \\
& x_{ij} \in \{0, 1\} && i = 1, \dots, n , \ j = 1, \dots, m
\end{align*}
$$

Rearranging the terms, we get:

$$
\begin{align*}
L_1(\lambda) : \quad \min_{x} \quad & \sum_{i=1}^{n} \sum_{j=1}^{m} (c_{ij} - \lambda_j)x_{ij} + \sum_{j=1}^{m} \lambda_j \\
\text{s.t.} \quad & \sum_{j=1}^{m} w_{ij}x_{ij} \le C_{i} && i = 1, \dots, n \\
& x_{ij} \in \{0, 1\} && i = 1, \dots, n , \ j = 1, \dots, m
\end{align*}
$$

This problem decomposes into $n$ independent subproblems, one for each agent. For each agent $i$, the subproblem is a 0-1 knapsack problem:

$$
\begin{align*}
\text{Subproblem } i: \quad \min_{x} \quad & \sum_{j=1}^{m} (c_{ij} - \lambda_j)x_{ij} \\
\text{s.t.} \quad & \sum_{j=1}^{m} w_{ij}x_{ij} \le C_{i} \\
& x_{ij} \in \{0, 1\} & j = 1, \dots, m
\end{align*}
$$

The Lagrangian dual problem is to maximize $L_1(\lambda)$ over all $\lambda$. This is done iteratively using the subgradient method. In each iteration, the Lagrange multipliers $\lambda$ are updated using the subgradient, which is given by $g_j = 1 - \sum_{i=1}^{n} x_{ij}$, where $x_{ij}$ is the solution to the knapsack subproblem for the current $\lambda$.

### Lagrangian Relaxation 2: Dualizing the Capacity Constraints

Alternatively, we can relax the agents' capacity constraints, introducing a vector of Lagrange multipliers $\mu \in \mathbb{R}^n, \mu \ge 0$.

The Lagrangian dual problem $L_2(\mu)$ is:

$$
\begin{align*}
L_2(\mu) : \quad \min_{x} \quad & \sum_{i=1}^{n} \sum_{j=1}^{m} c_{ij}x_{ij} + \sum_{i=1}^{n} \mu_i \left(\sum_{j=1}^{m} w_{ij}x_{ij} - C_{i}\right) \\
\text{s.t.} \quad & \sum_{i=1}^{n} x_{ij} = 1 && j = 1, \dots, m\\
& x_{ij} \in \{0, 1\} && i = 1, \dots, n, \ j = 1, \dots, m
\end{align*}
$$

Rearranging the terms:

$$
\begin{align*}
L_2(\mu) : \quad \min_{x} \quad & \sum_{j=1}^{m} \sum_{i=1}^{n} (c_{ij} + \mu_i w_{ij})x_{ij} - \sum_{i=1}^{n} \mu_i C_i \\
\text{s.t.} \quad & \sum_{i=1}^{n} x_{ij} = 1 && j = 1, \dots, m\\
& x_{ij} \in \{0, 1\} && i = 1, \dots, n, \ j = 1, \dots, m
\end{align*}
$$

This problem decomposes into $m$ independent subproblems, one for each job. For each job $j$, the subproblem is to find the agent that minimizes the adjusted cost:

$$
\text{Subproblem } j: \quad \min_{x} \quad \sum_{i=1}^{n} (c_{ij} + \mu_i w_{ij})x_{ij}
$$

subject to $\sum_{i=1}^{n} x_{ij} = 1$, which simply means finding $\min_{i} \{c_{ij} + \mu_i w_{ij}\}$.

The Lagrangian dual problem is to maximize $L_2(\mu)$ over $\mu \ge 0$. Again, the subgradient method is used, with the subgradient being $h_i = \sum_{j=1}^{m} w_{ij}x_{ij} - C_{i}$.

### Julia implementation

To provide a benchmark for comparison, we first solve the original problem to find the true optimal integer solution using JuMP. Then, we solve the first and second relaxation that provide lower bounds for the original problem.

In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 7 – Lagrangian relaxation
#  Section: Exercise 3
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP          # For mathematical programming
using HiGHS         # HiGHS solver
using LinearAlgebra # For norm calculations

# GAP Optimal Assignment Problem
function solve_gap_optimal(c,w,C)
    # Problem dimensions
    n, m = size(c)
    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)
    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)
    # Define binary variables
    @variable(model, x[1:n, 1:m], Bin)
    # Objective: minimize total assignment cost
    @objective(model, Min, sum(c[i, j] * x[i, j] for i in 1:n, j in 1:m))
    # Constraint: each job must be assigned to exactly one agent
    @constraint(model, [j in 1:m], sum(x[i, j] for i in 1:n) == 1)
    # Constraint: agent capacities
    @constraint(model, [i in 1:n], sum(w[i, j] * x[i, j] for j in 1:m) <= C[i])
    # Solve the problem
    JuMP.optimize!(model)
    # Display results
    println("\nOptimal assignment and costs:")
    for i in 1:n, j in 1:m
        if JuMP.value(x[i, j]) > 0.5
            println("Job $j assigned to Agent $i (Cost = $(c[i, j]), Workload = $(w[i, j]))")
        end
    end
    println("\nTotal Cost: ", JuMP.objective_value(model))
end

#--- Lagrangian Relaxation 1: Dualizing Assignment Constraints ---

# Solve the knapsack problem for each agent given the modified costs
function solve_knapsack(costs, weights, capacity)
    n_items = length(costs)
    model = JuMP.Model(HiGHS.Optimizer)
    JuMP.set_silent(model)
    @variable(model, y[1:n_items], Bin)
    @objective(model, Min, dot(costs, y))
    @constraint(model, dot(weights, y) <= capacity)
    JuMP.optimize!(model)
    return JuMP.objective_value(model), JuMP.value.(y)
end

# Subgradient method for Lagrangian Relaxation 1
function subgradient_lr1(c, w, C)
    n,m = size(c)
    λ = zeros(m)
    max_iter = 200
    best_lower_bound = -Inf

    for k in 1:max_iter
        # Solve subproblems (n knapsack problems)
        total_subproblem_obj = 0
        x_sol = zeros(n, m)
        for i in 1:n
            costs_i = c[i, :] .- λ
            weights_i = w[i, :]
            obj_i, x_i = solve_knapsack(costs_i, weights_i, C[i])
            total_subproblem_obj += obj_i
            x_sol[i, :] = x_i
        end

        lower_bound = sum(λ) + total_subproblem_obj
        best_lower_bound = max(best_lower_bound, lower_bound)

        # Subgradient update
        g = 1 .- sum(x_sol, dims=1)[:]
        if norm(g) < 1e-6
            break
        end

        α = 1.0 / k # Step size
        λ .+= α .* g
    end
    return best_lower_bound
end

#--- Lagrangian Relaxation 2: Dualizing Capacity Constraints ---

# Subgradient method for Lagrangian Relaxation 2
function subgradient_lr2(c, w, C)
    n,m = size(c)
    μ = zeros(n)
    max_iter = 200
    best_lower_bound = -Inf

    for k in 1:max_iter
        # Solve subproblems (m simple assignment problems)
        total_subproblem_obj = 0
        x_sol = zeros(n, m)
        for j in 1:m
            costs_j = c[:, j] + μ .* w[:, j]
            min_cost, min_idx = findmin(costs_j)
            total_subproblem_obj += min_cost
            x_sol[min_idx, j] = 1
        end

        lower_bound = -dot(μ, C) + total_subproblem_obj
        best_lower_bound = max(best_lower_bound, lower_bound)

        # Subgradient update
        h = sum(w .* x_sol, dims=2)[:] - C
        if norm(h) < 1e-6
            break
        end

        α = 1.0 / k # Step size
        μ = max.(0, μ + α .* h)
    end
    return best_lower_bound
end

# Cost matrix c[i, j]
c = [
    8  6 10  9  7;
    9 12  7  8 10;
   14 10  9  7  8
]

# Workload matrix w[i, j]
w = [
    2 4 3 2 5;
    3 2 4 3 2;
    5 3 2 4 3
]

# Capacity for each agent
C = [10, 9, 11]

# Run everything
println("Solving GAP Optimal Assignment Problem:")
solve_gap_optimal(c, w, C)

println("\n--- Lagrangian Relaxation ---")
lr1_bound = subgradient_lr1(c, w, C)
println("Lower Bound from Relaxation 1: ", lr1_bound)

lr2_bound = subgradient_lr2(c, w, C)
println("Lower Bound from Relaxation 2: ", lr2_bound)

Solving GAP Optimal Assignment Problem:

Optimal assignment and costs:
Job 1 assigned to Agent 1 (Cost = 8, Workload = 2)
Job 2 assigned to Agent 1 (Cost = 6, Workload = 4)
Job 3 assigned to Agent 2 (Cost = 7, Workload = 4)
Job 4 assigned to Agent 3 (Cost = 7, Workload = 4)
Job 5 assigned to Agent 3 (Cost = 8, Workload = 3)

Total Cost: 36.0

--- Lagrangian Relaxation ---
Lower Bound from Relaxation 1: 29.36515474060723
Lower Bound from Relaxation 2: 35.2
